# Running the tests on a real star

Every test is a standalone command-line tool with the same vocabulary. None of
them imports another; they communicate through JSON files on disk, so you can
run one, look at it, and run the next.

This notebook uses **DS Tuc A** — a 45 Myr K star in a wide binary with a
known 8.14-day planet. The binary is the point: a companion 5 arcseconds away
is the kind of thing that makes several of these tests interesting.

**You need a light curve.** Either set `PETCARE_LC_ROOTS` to a directory tree
containing one (the loader searches by TIC and sniffs `.npz`, CSV and FITS), or
set `LC_FILES` below to explicit paths. If you have neither, the cells will
say so and stop rather than fail obscurely.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

TIC = 410214986
NAME = "DS Tuc A"
# planet b, from ../examples/ds_tuc/candidates.csv; planet c is masked as a known signal
PERIOD, T0, DURATION_H = 8.1382107397, 1332.30989489, 3.197
KNOWN = "DS_Tuc_A_c:21.6248983386:1329.19170623:3.772"

OUT = Path("out_notebook")
OUT.mkdir(exist_ok=True)

# the light curves shipped with the DS Tuc example (made by pc-extract);
# point this at your own files, or leave it empty to search $PETCARE_LC_ROOTS
LC_FILES = sorted(Path("../examples/ds_tuc/data").glob("*/*.npz"))

TARGET = ["--tic", str(TIC), "--name", NAME, "--age", "45", "--known", KNOWN]
if LC_FILES:
    TARGET += ["--lc", *[str(p) for p in LC_FILES]]

def run(tool, *args, show=True):
    cmd = [sys.executable, "-m", f"petcare.scripts.{tool}",
           *TARGET, "--out", str(OUT), *[str(a) for a in args]]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if show:
        print(p.stdout.strip() or p.stderr.strip()[-2000:])
    return p

print(len(LC_FILES), "light-curve files; roots:", os.environ.get("PETCARE_LC_ROOTS", "(unset)"))
print("stellar catalogue:", os.environ.get("PETCARE_STELLAR_CATALOG", "(unset)"))

## 1. The fit comes first

`pc_fit` refines the ephemeris and measures the depth, duration and impact
parameter. The other light-curve tests can then read that refined ephemeris
with `--use-fit` instead of the input one, which matters: a candidate period
from a box search is routinely off by enough to smear the fold.

In [ ]:
p = run("pc_fit", "--period", PERIOD, "--t0", T0, "--duration", DURATION_H)

## 2. What a result record looks like

Every test writes the same schema. The parts that carry weight:

- **`status`** — `PASS`, `FLAG`, `FAIL`, `INCONCLUSIVE` or `ERROR`. A test
  fails a candidate only when it refutes it *by itself*; anything it merely
  cannot answer is `INCONCLUSIVE`, which is neither a pass nor a fail.
- **`self_check`** — what the test did to prove it measured what it claims.
  A test whose self-check fails is not allowed to report a detection.
- **`noise`** — the level a null result rules out at. A non-detection without
  this number means nothing.
- **`provenance`** — which light curve, how many cadences, which commit.

In [ ]:
fit = json.loads((OUT / f"TIC{TIC}" / "fit.json").read_text())

print(fit["status"], "|", fit["reason"], "\n")
print("self-check:", fit["self_check"])
print("noise:     ", fit["noise"])
print("provenance:", {k: fit["provenance"][k] for k in
                      list(fit["provenance"])[:5]})

m = fit["metrics"]
for k in ("period", "t0_btjd", "depth_ppm", "duration_h", "b", "a_rs",
          "rp_earth", "rho_star_gcc", "rho_transit_gcc"):
    v = m.get(k)
    print(f"  {k:18s} {v if not isinstance(v, float) else round(v, 5)}")

The fit also records **which variability model it used and why**. On an active
star a global Gaussian process can explain the transits itself; `pc_fit` runs
a local-baseline fit alongside it and keeps whichever the two agree on.

In [ ]:
print(json.dumps(m["variability_model"], indent=1)[:1500])

## 3. The rest of the light-curve tests

Each reads the refined ephemeris with `--use-fit`. They are independent: run
them in any order, or only the ones you care about.

In [ ]:
TESTS = ["pc_depths", "pc_oddeven", "pc_shape", "pc_duration",
         "pc_secondary", "pc_rotation", "pc_flares", "pc_instrument"]
for tool in TESTS:
    run(tool, "--use-fit", show=False)
    print(f"  {tool} done")

In [ ]:
rows = []
for f in sorted((OUT / f"TIC{TIC}").glob("*.json")):
    if f.name == "petcare.json":
        continue
    d = json.loads(f.read_text())
    rows.append((d["test"], d["status"], d["reason"]))

w = max(len(r[0]) for r in rows)
for test, status, reason in sorted(rows, key=lambda r: r[1]):
    print(f"{test:{w}s}  {status:13s}  {reason[:96]}")

Read that column, not a score. **PetCare deliberately computes no overall
number**: which test objected is the information, and a scalar hides it. The
standing preference on this project is to let a few bad candidates through and
disprove them with follow-up, rather than to build an aggressive automatic cut.

Note what `pc_oddeven` and `pc_rotation` do *not* do — neither can return
`FAIL` at any significance. At the textbook 3σ, odd/even would delete 18% of
the known young planets in this sample, because spot crossings make epoch
depths vary far beyond their formal errors. Rotation proximity as a veto
previously deleted 25 signals including one published as a planet.

## 4. The figures

Every test draws exactly the plot a human needs to check its claim. For the
fit that is the variability model over the raw data — if it is tracking the
transits rather than the star, no formal error will tell you.

In [ ]:
from IPython.display import Image, display

for plot in fit.get("plots", []):
    p = Path(plot)
    if p.exists():
        display(Image(filename=str(p)))

In [ ]:
for test in ("odd_even", "depths", "secondary"):
    p = OUT / f"TIC{TIC}" / f"{test}.png"
    if p.exists():
        print(test)
        display(Image(filename=str(p)))

## 5. Your own verdict

The record has a `human` block reserved for you. Re-running a test preserves
it and files the machine's previous answer under `previous`, so a disposition
you wrote by hand is never overwritten by a later run.

In [ ]:
path = OUT / f"TIC{TIC}" / "odd_even.json"

record = json.loads(path.read_text())
record["human"] = {
    "disposition": "planet", "by": "you", "date": "2026-09-21",
    "note": "depth scatter is spot crossings; consistent across sectors"}
path.write_text(json.dumps(record, indent=1))

# now run the test again -- the disposition survives
run("pc_oddeven", "--use-fit", show=False)
after = json.loads(path.read_text())
print("human: ", after["human"])
print("machine's previous answer, filed under 'previous':",
      (after.get("previous") or {}).get("status"))

Next: [`03_a_full_vetting_run.ipynb`](03_a_full_vetting_run.ipynb) — the
driver, a candidate table, and the cross-target tests.